# 00 — Setup the official RiemannGFM baseline

Run this once at the start of each Colab session. It:
1. Verifies GPU is attached (Runtime -> Change runtime type -> T4 GPU)
2. Clones `github.com/RiemannGraph/RiemannGFM` into `/content/RiemannGFM_official/`
3. Installs the exact package versions the paper's authors used
4. Mounts Google Drive so checkpoints and datasets survive session resets

After this notebook completes you can run 01–04 in any order (they only depend on 01 being run first for pretraining).

In [ ]:
# 1. GPU check.
!nvidia-smi | head -n 15

In [ ]:
# 2. Clone the official repository.
import os
OFFICIAL_DIR = '/content/RiemannGFM_official'
if not os.path.exists(OFFICIAL_DIR):
    !git clone https://github.com/RiemannGraph/RiemannGFM.git $OFFICIAL_DIR
else:
    print(f'{OFFICIAL_DIR} already exists — pulling latest')
    !git -C $OFFICIAL_DIR pull --ff-only

In [ ]:
# 3. Show what's inside the official repo.
!ls -la $OFFICIAL_DIR

In [ ]:
# 4. Install pinned dependencies from the official requirements.txt.
# torch is already on Colab; the pin below aligns cuda version.
!pip install --quiet torch==2.0.0 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# torch_scatter must match the torch build.
!pip install --quiet torch_scatter -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

In [ ]:
# Remaining requirements, part 1 — critical packages that must match the official repo's API.
# Installed in their own command so a failure here is loud and doesn't get masked by anything else.
!pip install --quiet torch_geometric==2.6.1 geoopt==0.5.0 ogb==1.3.6

In [ ]:
# Remaining requirements, part 2 — auxiliary utility packages.
# Deliberately unpinned: the old pins here (numpy==1.24.2, scipy==1.10.1, tqdm==4.61.2, ...)
# have no wheels for current Colab's Python version, and bundling them with torch_geometric
# in one pip command previously caused the WHOLE command (including torch_geometric) to
# silently fail to install. Split + unpinned so this can't block the critical packages above.
!pip install --quiet gensim matplotlib networkx numpy scikit-learn scipy tqdm pyyaml

In [ ]:
# 5. Mount Drive for persistent artefacts.
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/RiemannGFM/baseline'
os.makedirs(f'{BASE}/checkpoints', exist_ok=True)
os.makedirs(f'{BASE}/datasets', exist_ok=True)
os.makedirs(f'{BASE}/results', exist_ok=True)
print(f'Drive workspace: {BASE}')

In [ ]:
# 6. Wire the official repo's datasets/ and checkpoints/ dirs to Drive so downloads persist.
# The official code hard-codes these paths.
import os
for name in ['datasets', 'checkpoints']:
    link = os.path.join(OFFICIAL_DIR, name)
    target = os.path.join(BASE, name)
    if os.path.islink(link) or os.path.exists(link):
        !rm -rf $link
    os.symlink(target, link)
    print(f'{link} -> {target}')

In [ ]:
# 7. Sanity — versions and files ready.
import sys, torch, torch_geometric, geoopt
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda_available', torch.cuda.is_available())
print('pyg', torch_geometric.__version__)
print('geoopt', geoopt.__version__)
print()
print('Official repo scripts:')
!ls $OFFICIAL_DIR/scripts
!ls $OFFICIAL_DIR/scripts/NC
!ls $OFFICIAL_DIR/scripts/LP

**Done.** Move on to `01_pretrain.ipynb`.